In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
import math
from typing import Iterable, Optional

import numpy as np
import pandas as pd
from numpy.polynomial.chebyshev import chebvander

from .basis import (
    alpha_norm,
    block_design,
    rbf_basis,
    second_difference_matrix,
    tensor_features,
    theta_from_x,
)


@dataclass
class PCSSORParams:
    """Hyperparameters for PC-SSOR."""

    K: int = 9
    M: int = 25
    sigma_scale: float = 1.5
    lam: float = 1e-6
    eta: float = 1e-3
    mu_sym: float = 10.0
    sym_grid: int = 40
    alpha_clip: bool = True


class PCSSOR:
    """Physics-Constrained Spectral Surrogate Operator Regression.

    Model form:
      Cp_s(x, alpha) = sum_k sum_m w_{s,k,m} T_k(alpha_tilde) phi_m(theta(x))

    Constraints/regularization:
      - ridge stabilization
      - chordwise second-difference smoothness
      - soft upper/lower symmetry at alpha = 0
      - hard trailing-edge continuity for each Chebyshev mode
    """

    def __init__(self, params: PCSSORParams | None = None):
        self.params = params or PCSSORParams()
        self.model_: dict | None = None

    @staticmethod
    def _solve_kkt(H: np.ndarray, g: np.ndarray, A: np.ndarray, b: np.ndarray) -> np.ndarray:
        d = H.shape[0]
        m = A.shape[0]
        KKT = np.zeros((d + m, d + m), dtype=np.float64)
        KKT[:d, :d] = H
        KKT[:d, d:] = A.T
        KKT[d:, :d] = A
        rhs = np.concatenate([g, b])
        sol = np.linalg.solve(KKT, rhs)
        return sol[:d]

    def fit(self, df_train: pd.DataFrame) -> "PCSSOR":
        p = self.params
        df_train = df_train.copy()
        alpha_min = float(df_train["Alpha"].min())
        alpha_max = float(df_train["Alpha"].max())
        alpha_span = max(alpha_max - alpha_min, 1e-12)

        theta_train = theta_from_x(df_train["x"].values)
        theta_min = float(theta_train.min())
        theta_max = float(theta_train.max())

        centers = np.linspace(theta_min, theta_max, p.M - 1)
        if len(centers) > 1:
            spacing = float(centers[1] - centers[0])
        else:
            spacing = max(theta_max - theta_min, 1e-6)
        sigma = max(1e-6, p.sigma_scale * spacing)

        model_like = {
            "K": p.K,
            "centers": centers,
            "sigma": sigma,
            "alpha_min": alpha_min,
            "alpha_max": alpha_max,
            "alpha_span": alpha_span,
            "alpha_clip": p.alpha_clip,
        }

        F, Mtot = tensor_features(df_train, model_like)
        P = F.shape[1]
        X = block_design(F, df_train["side"].values)
        y = df_train["Cp"].values.astype(np.float64)

        H = X.T @ X + p.lam * np.eye(2 * P, dtype=np.float64)
        g = X.T @ y

        if p.eta and p.eta > 0:
            dtd = second_difference_matrix(Mtot, exclude_bias=True)
            R_side = np.kron(np.eye(p.K + 1), dtd)
            R = np.zeros((2 * P, 2 * P), dtype=np.float64)
            R[:P, :P] = R_side
            R[P:, P:] = R_side
            H = H + p.eta * R

        if p.mu_sym and p.mu_sym > 0:
            theta_grid = np.linspace(theta_min, theta_max, p.sym_grid)
            phi_grid = rbf_basis(theta_grid, centers, sigma, add_bias=True)
            T0 = chebvander(
                alpha_norm(np.array([0.0]), alpha_min, alpha_span, clip=p.alpha_clip),
                p.K,
            ).ravel()
            Fg = np.einsum("k,lm->lkm", T0, phi_grid).reshape(len(theta_grid), -1)
            Xsym = np.zeros((len(theta_grid), 2 * P), dtype=np.float64)
            Xsym[:, :P] = Fg
            Xsym[:, P:] = -Fg
            weight = math.sqrt(p.mu_sym)
            H = H + (weight * Xsym).T @ (weight * Xsym)

        x_te = float(df_train["x"].max())
        theta_te = float(theta_from_x(np.array([x_te]))[0])
        phi_te = rbf_basis(np.array([theta_te]), centers, sigma, add_bias=True).ravel()
        A_rows = []
        for k in range(p.K + 1):
            row = np.zeros(2 * P, dtype=np.float64)
            row[k * Mtot : (k + 1) * Mtot] = phi_te
            row[P + k * Mtot : P + (k + 1) * Mtot] = -phi_te
            A_rows.append(row)
        A = np.vstack(A_rows)
        b = np.zeros(A.shape[0], dtype=np.float64)
        w = self._solve_kkt(H, g, A, b)

        self.model_ = {
            "name": "PC-SSOR (Physics-Constrained Spectral Surrogate Operator Regression)",
            "K": p.K,
            "M": p.M,
            "Mtot": Mtot,
            "centers": centers,
            "sigma": sigma,
            "x_te": x_te,
            "alpha_min": alpha_min,
            "alpha_max": alpha_max,
            "alpha_span": alpha_span,
            "alpha_clip": p.alpha_clip,
            "params": asdict(p),
            "w": w,
        }
        return self

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        if self.model_ is None:
            raise RuntimeError("Model is not fitted. Call fit() first.")
        df = df.copy()
        if "side" not in df.columns:
            df["side"] = np.where(df["y"].values >= 0.0, "upper", "lower")
        F, _ = tensor_features(df, self.model_)
        X = block_design(F, df["side"].values)
        return (X @ self.model_["w"]).astype(np.float64)

    def te_gap(self, alpha_values) -> float:
        if self.model_ is None:
            raise RuntimeError("Model is not fitted. Call fit() first.")
        rows = []
        for alpha in sorted(np.unique(alpha_values)):
            rows.append({"x": self.model_["x_te"], "y": +1e-8, "Alpha": float(alpha), "side": "upper"})
            rows.append({"x": self.model_["x_te"], "y": -1e-8, "Alpha": float(alpha), "side": "lower"})
        te_df = pd.DataFrame(rows)
        pred = self.predict(te_df)
        te_df["Cp_pred"] = pred
        gaps = []
        for alpha in sorted(te_df["Alpha"].unique()):
            sub = te_df[te_df["Alpha"] == alpha]
            cu = float(sub[sub["side"] == "upper"]["Cp_pred"].iloc[0])
            cl = float(sub[sub["side"] == "lower"]["Cp_pred"].iloc[0])
            gaps.append(abs(cu - cl))
        return float(np.mean(gaps)) if gaps else float("nan")

    def to_dict(self) -> dict:
        if self.model_ is None:
            raise RuntimeError("Model is not fitted. Call fit() first.")
        return self.model_.copy()

    @classmethod
    def from_dict(cls, payload: dict) -> "PCSSOR":
        model = cls(PCSSORParams(**payload.get("params", {})))
        model.model_ = payload
        return model
